# 70 — Generate the overlay T-Box

The authored qualified-BC layer as OWL, generated from `overlay/qbc.schema.yaml`.

Runs **after** `10_fetch_cosmos.ipynb` and `20_generate.ipynb`: the schema imports
`build/cosmos_bc_model.patched.yaml`, this repo's own repaired copy of the pinned
BC model. Against the unrepaired copy the `cosmos_bc` prefix carries no separator
(`known-gaps.md` §1a) and the class-level `skos:broadMatch` resolves to
`...biomedical_concept_v1.0BiomedicalConcept`, which is in no graph — measured
2026-09-01, and the reason the overlay source could not simply be copied across.

Decisions exercised here: **D13** (the `qbc` namespace), **D19** (the SDTM model is
not imported; `AssignedTerm` and `domain` are local stand-ins), **D6** (every
generator option pinned explicitly), **D7** (this is the first deliverable that
names *terms* under w3id, and it does so deliberately), **D9** (canonicalize
before writing).

## Configuration

In [1]:
DOWNLOADS = "../downloads"
OVERLAY   = "../overlay"
ROOT      = ".."

SCHEMA = f"{OVERLAY}/qbc.schema.yaml"
TARGET = "cosmos_qbc_v1.ttl"

QBC_NS = "https://w3id.org/cdisc/cosmos/qbc/"
BC_NS  = "https://www.cdisc.org/cosmos/biomedical_concept_v1.0/"

CORE_BC_TTL = f"{ROOT}/cosmos_bc_v1.ttl"

VERSION      = "0.2.0"
ONTOLOGY_IRI = QBC_NS
CORE_BC_IRI  = "https://w3id.org/cdisc/cosmos/bc/"

## Generate

Same pinned option set as `20_generate.ipynb`, with one change: **`mergeimports`
is `False`**.

With imports merged, `gen-owl` redeclares the imported CDISC enums —
`BiomedicalConceptResultScaleEnum`, `DataElementConceptDataTypeEnum`,
`PackageTypeEnum` — under this schema's namespace, i.e. it renames three terms
CDISC published. That is precisely what decision D7 declined to do. Unmerged, the
overlay T-Box declares only its own nine classes and references CDISC's terms at
CDISC's IRIs.

What that does not fix: the `rdfs:range` of `qbc:resultScale` still reads
`qbc:BiomedicalConceptResultScaleEnum`, now undeclared, because `gen-owl` names
an imported enum after the importing schema either way. The A-Box uses the core
enum IRIs regardless (decision D20), so this is a divergence to classify
alongside D11's, not a range to satisfy.

In [2]:
from pathlib import Path

from linkml.generators.owlgen import MetadataProfile, OwlSchemaGenerator
from rdflib import Graph

# Pinned to the gen-owl CLI behaviour at linkml 1.11.1, per decision D6.
# Identical to 20_generate.ipynb except mergeimports.
OWL_OPTIONS = {
    "metadata_profiles": [MetadataProfile.linkml],
    "metadata": True,
    "mergeimports": False,
    "metaclasses": False,
    "type_objects": False,
    "ontology_uri_suffix": ".owl.ttl",
    "add_ols_annotations": True,
    "add_root_classes": False,
    "assert_equivalent_classes": False,
    "enum_iri_separator": "#",
    "enum_inherits_as_subclass_of": False,
    "mixins_as_expressions": False,
    "skip_abstract_class_as_unionof_subclasses": False,
    "use_native_uris": True,
    "useuris": True,
    "xsd_anyuri_as_iri": False,
    "default_permissible_value_type": "http://www.w3.org/2002/07/owl#Class",
    "skip_vacuous_min_zero_cardinality_axioms": False,
    "skip_vacuous_local_range_axioms": False,
    "consolidate_cardinality_axioms": False,
}

generator = OwlSchemaGenerator(SCHEMA, **OWL_OPTIONS)
graph = Graph().parse(data=generator.serialize(), format="turtle")

print(f"{TARGET:20s} {len(graph):>6,} triples generated  <- {SCHEMA}")

Ambiguous attribute: conceptId https://w3id.org/cdisc/cosmos/qbc/conceptId
Ambiguous attribute: value https://w3id.org/cdisc/cosmos/qbc/value
Ambiguous attribute: conceptId https://w3id.org/cdisc/cosmos/qbc/conceptId
Ambiguous attribute: value https://w3id.org/cdisc/cosmos/qbc/value
Ambiguous attribute: sourceAnchor https://w3id.org/cdisc/cosmos/qbc/sourceAnchor
Ambiguous attribute: sourceAnchor https://w3id.org/cdisc/cosmos/qbc/sourceAnchor


cosmos_qbc_v1.ttl       638 triples generated  <- ../overlay/qbc.schema.yaml


`gen-owl` logs `Ambiguous attribute` for `conceptId`, `value` and `sourceAnchor`.
Each is declared on more than one class in the overlay schema, LinkML collapses
them onto one property IRI, and `gen-owl` responds by emitting a bare
`owl:DatatypeProperty` with no range and no pattern. `gen-shacl` keeps the
per-class patterns apart — `AssignedTerm.conceptId` `^(C[0-9]+|CNEW)$` against
`ConceptTerm.conceptId` `^(C[0-9]+)$`.

The warning is left in place. Attaching `slot_uri` silences it, but silencing is
all it does: measured 2026-09-01, the two attributes still resolve to one IRI and
their definitions are merged onto it instead of dropped. A loud collapse is worth
more than a quiet one. This is decision D1's finding at overlay scale, in a schema
this repo wrote.

## Author the ontology header

Same act as `20_generate.ipynb` cell 8: the generated `owl:Ontology` node is
replaced, because left alone `ontology_uri_suffix` would name this ontology
`https://w3id.org/cdisc/cosmos/qbc.owl.ttl` — a file name standing in for an
ontology IRI.

Two differences from the core headers. There is no `dcterms:source`: nothing was
fetched, the schema is authored in this repo, and pointing `source` at CDISC's
model would misattribute it. And the generated `owl:imports` is the literal
relative import path, so it is replaced by the core BC ontology IRI — which is
what a consumer actually needs to resolve.

In [3]:
import json

from rdflib import Literal, Namespace, URIRef
from rdflib.namespace import DCTERMS, OWL, RDF, RDFS, SKOS, XSD

VANN = Namespace("http://purl.org/vocab/vann/")

CREATED  = "2026-09-01"
MODIFIED = "2026-09-02"
CREATOR  = "Kerstin Forsberg"
LICENSE  = "https://opensource.org/licenses/MIT"

TITLE = "CDISC COSMoS Qualified Biomedical Concepts (overlay)"

DESCRIPTION = (
    "An authored overlay on the mechanical rendering of CDISC COSMoS: the "
    "qualified (sibling) biomedical concept, with an identity of its own, typed "
    "mappings to external code systems, a single result scale, an authoritative "
    "result data element concept, a concept-level semantic value set, and a "
    "state-keyed interpretation-regime assertion. Every term declared here is "
    "this repo's, not CDISC's; the terms it reuses keep the IRIs CDISC published. "
    "The concepts it qualifies are the ones cosmos_bc_v1.instances.ttl renders, "
    "reached by skos:broader."
)

COMMENT = (
    "Authored overlay, not a rendering of published COSMoS. "
    "Draft - not a normative CDISC artifact."
)

ANNOTATION_PROPERTIES = [
    DCTERMS.title,
    DCTERMS.description,
    DCTERMS.creator,
    DCTERMS.created,
    DCTERMS.modified,
    DCTERMS.license,
    DCTERMS.identifier,
    DCTERMS.bibliographicCitation,
    VANN.preferredNamespacePrefix,
    VANN.preferredNamespaceUri,
]

In [4]:
meta = json.loads(Path(DOWNLOADS, ".fetch_meta_bc_export.json").read_text(encoding="utf-8"))

existing = list(graph.subjects(RDF.type, OWL.Ontology))
if len(existing) != 1:
    raise RuntimeError(f"expected 1 owl:Ontology node, found {len(existing)}")
graph.remove((existing[0], None, None))

ontology = URIRef(ONTOLOGY_IRI)
graph.add((ontology, RDF.type, OWL.Ontology))
graph.add((ontology, OWL.imports, URIRef(CORE_BC_IRI)))
graph.add((ontology, RDFS.label, Literal(TITLE)))
graph.add((ontology, RDFS.comment, Literal(COMMENT)))
graph.add((ontology, DCTERMS.title, Literal(TITLE)))
graph.add((ontology, DCTERMS.description, Literal(DESCRIPTION)))
graph.add((ontology, DCTERMS.creator, Literal(CREATOR)))
graph.add((ontology, DCTERMS.created, Literal(CREATED, datatype=XSD.date)))
graph.add((ontology, DCTERMS.modified, Literal(MODIFIED, datatype=XSD.date)))
graph.add((ontology, DCTERMS.license, URIRef(LICENSE)))
graph.add((ontology, DCTERMS.identifier, Literal(meta["package_date"])))
graph.add((
    ontology,
    DCTERMS.bibliographicCitation,
    Literal(f"Forsberg, K. ({CREATED[:4]}). {TITLE}. {ONTOLOGY_IRI}"),
))
graph.add((ontology, OWL.versionIRI, URIRef(ONTOLOGY_IRI + VERSION)))
graph.add((ontology, OWL.versionInfo, Literal(f"v{VERSION}")))
graph.add((ontology, VANN.preferredNamespacePrefix, Literal("qbc")))
graph.add((ontology, VANN.preferredNamespaceUri, URIRef(QBC_NS)))

for annotation in ANNOTATION_PROPERTIES:
    graph.add((annotation, RDF.type, OWL.AnnotationProperty))

print(f"{TARGET:20s} {len(graph):>6,} triples  ontology {ONTOLOGY_IRI}  versionIRI {ONTOLOGY_IRI}{VERSION}")

cosmos_qbc_v1.ttl       660 triples  ontology https://w3id.org/cdisc/cosmos/qbc/  versionIRI https://w3id.org/cdisc/cosmos/qbc/0.2.0


## Repair the imported-enum range — decision D20

The one authored edit to generated *terms* in this notebook, and it is named
rather than silent. It runs after the header is authored, so that every edit to
the generated graph happens in one place — and because the guard below would
otherwise trip on the generated `owl:imports`, which is the literal relative
import path until the header replaces it.

`gen-owl` names an imported enum after the **importing** schema, in either merge
mode. So `qbc:resultScale` comes out with `rdfs:range
qbc:BiomedicalConceptResultScaleEnum` — an IRI declared nowhere, while the A-Box
emits the core permissible values (D20). Measured 2026-09-01: exactly one range
in this graph dangles this way.

It is retargeted to the enum CDISC published, which the core T-Box declares as
`owl:unionOf` over exactly its five permissible values. That is the idiom
`usdm-rdf` uses for a multi-target range — `rdfs:range [ a owl:Class ;
owl:unionOf ( … ) ]`, with its `examples/05_polymorphic_associations.ipynb`
showing the SPARQL that reads it — reached here by *pointing at* the union
instead of copying it, so the overlay cannot drift from core and no CDISC term
is renamed.

Worth recording why the problem exists here and not in `usdm-rdf`: a USDM coded
attribute carries `usdm:boundCodelist` to the codelist's NCIt C-code and lets
NCIt name the values. Every COSMoS enum is bare — no `meaning:`, no `code_set`,
no C-code — so the generator's invented IRIs are the only identity a result
scale has anywhere. Same gap as `known-gaps.md` §4, one layer down.

In [5]:
from rdflib import URIRef
from rdflib.collection import Collection
from rdflib.namespace import OWL, RDF, RDFS

# Property local name -> the imported enum it should range over.
REPAIR = {
    "resultScale": "BiomedicalConceptResultScaleEnum",
}


def undeclared_qbc_iris(target, ignore=frozenset()):
    """qbc: IRIs referenced as an object but never appearing as a subject."""
    return {o for triple in target for o in triple
            if isinstance(o, URIRef) and str(o).startswith(QBC_NS)
            and o not in ignore and (o, None, None) not in target}


core_tbox = Graph().parse(CORE_BC_TTL, format="turtle")

# The version IRI is a release identifier, not a term, so it is never a subject.
VERSION_IRI = URIRef(ONTOLOGY_IRI + VERSION)

# Detect in ANY object position, not just rdfs:range: gen-owl also names the
# enum inside the owl:Restriction it puts on the declaring class, and a repair
# that fixed only the range would leave that second pointer dangling.
dangling = undeclared_qbc_iris(graph, ignore={VERSION_IRI})
expected = {URIRef(QBC_NS + enum_name) for enum_name in REPAIR.values()}
if dangling != expected:
    raise RuntimeError(f"the set of dangling qbc: IRIs changed: {sorted(map(str, dangling))}")

for prop, enum_name in REPAIR.items():
    old_iri = URIRef(QBC_NS + enum_name)
    new_iri = URIRef(BC_NS + enum_name)

    union = core_tbox.value(new_iri, OWL.unionOf)
    if union is None:
        raise RuntimeError(f"{new_iri} is not declared as a union in the core T-Box")
    members = list(Collection(core_tbox, union))

    positions = list(graph.triples((None, None, old_iri)))
    for s, predicate, o in positions:
        graph.remove((s, predicate, o))
        graph.add((s, predicate, new_iri))

    print(f"qbc:{prop}  {old_iri}")
    print(f"{'':13s}-> {new_iri}  (owl:unionOf over {len(members)} permissible values)")
    for member in sorted(members, key=str):
        print(f"{'':16s}{member}")
    print(f"{'':13s}replaced in {len(positions)} position(s): "
          f"{sorted(str(predicate).rsplit('#', 1)[-1] for _, predicate, _ in positions)}")

if undeclared_qbc_iris(graph, ignore={VERSION_IRI}):
    raise RuntimeError("a qbc: IRI is still referenced but never declared")

qbc:resultScale  https://w3id.org/cdisc/cosmos/qbc/BiomedicalConceptResultScaleEnum
             -> https://www.cdisc.org/cosmos/biomedical_concept_v1.0/BiomedicalConceptResultScaleEnum  (owl:unionOf over 5 permissible values)
                https://www.cdisc.org/cosmos/biomedical_concept_v1.0/BiomedicalConceptResultScaleEnum#Narrative
                https://www.cdisc.org/cosmos/biomedical_concept_v1.0/BiomedicalConceptResultScaleEnum#Nominal
                https://www.cdisc.org/cosmos/biomedical_concept_v1.0/BiomedicalConceptResultScaleEnum#Ordinal
                https://www.cdisc.org/cosmos/biomedical_concept_v1.0/BiomedicalConceptResultScaleEnum#Quantitative
                https://www.cdisc.org/cosmos/biomedical_concept_v1.0/BiomedicalConceptResultScaleEnum#Temporal
             replaced in 2 position(s): ['allValuesFrom', 'range']


## Canonicalize before writing — decision D9

In [6]:
from rdflib.compare import isomorphic, to_canonical_graph


def canonicalize(source):
    canonical = to_canonical_graph(source)
    if not isomorphic(canonical, source):
        raise RuntimeError("canonicalization changed the graph")
    if len(canonical) != len(source):
        raise RuntimeError(f"canonicalization changed triple count: {len(source)} -> {len(canonical)}")

    result = Graph()
    for triple in canonical:
        result.add(triple)

    result.bind("qbc", QBC_NS)
    result.bind("cosmos_bc", BC_NS)
    result.bind("cosmos_sdtm", "https://www.cdisc.org/cosmos/sdtm_v1.0/")
    result.bind("dcterms", DCTERMS)
    result.bind("vann", VANN)
    result.bind("skos", SKOS)
    result.bind("linkml", "https://w3id.org/linkml/")
    return result


canonical = canonicalize(graph)
turtle = canonical.serialize(format="turtle")

again = canonicalize(Graph().parse(data=turtle, format="turtle")).serialize(format="turtle")
if again != turtle:
    raise RuntimeError(f"{TARGET}: canonical serialization is not stable")

Path(ROOT, TARGET).write_text(turtle, encoding="utf-8")
print(f"{TARGET:32s} {len(canonical):>6,} triples  {len(turtle):>7,} chars  stable")

cosmos_qbc_v1.ttl                   660 triples   26,961 chars  stable


## Confirm what was generated

Four assertions, all fail-fast.

1. The overlay declares exactly the nine classes the schema does, and redeclares
   no CDISC term.
2. The class-level `skos:broadMatch` that `broad_mappings` generates lands on a
   class the core T-Box declares. This is the machine-readable form of the
   own-class decision — `QualifiedBiomedicalConcept` is not `is_a`
   `BiomedicalConcept`, it *broad-matches* it — and it is the join to core at the
   schema level. Note it is a **class-level** statement; the instance-level
   analyte link is `skos:broader` (decision D14).
3. The terms this deliverable names under w3id are the overlay's own, and no
   CDISC term has moved into the `qbc` namespace.
4. The CDISC terms the overlay references are all present in the core T-Box.

In [7]:
EXPECTED_CLASSES = {
    "AssignedTerm",
    "ConceptTerm",
    "ExternalMapping",
    "InterpretationRegimeAssertion",
    "MappingRelationEnum",
    "QualifiedBiomedicalConcept",
    "QualifiedBiomedicalConceptCollection",
    "Recording",
    "SemanticValueSetTerm",
}

written = Graph().parse(Path(ROOT, TARGET), format="turtle")

declared = {str(s).replace(QBC_NS, "") for s in written.subjects(RDF.type, OWL.Class)
            if str(s).startswith(QBC_NS) and "#" not in str(s)}
if declared != EXPECTED_CLASSES:
    raise RuntimeError(f"declared classes changed: {declared ^ EXPECTED_CLASSES}")
print(f"ok    classes declared: {len(declared)}")

broad = [o for o in written.objects(URIRef(QBC_NS + "QualifiedBiomedicalConcept"), SKOS.broadMatch)]
if broad != [URIRef(BC_NS + "BiomedicalConcept")]:
    raise RuntimeError(f"class-level broadMatch is {broad}")
if (broad[0], RDF.type, OWL.Class) not in core_tbox:
    raise RuntimeError(f"{broad[0]} is not declared in the core T-Box")
print(f"ok    class-level skos:broadMatch -> {broad[0]}  (present in core)")

cdisc = sorted({str(n) for triple in written for n in triple
                if str(n).startswith("https://www.cdisc.org/")})
missing = [t for t in cdisc if (URIRef(t), None, None) not in core_tbox
           and not t.startswith("https://www.cdisc.org/cosmos/sdtm_v1.0/")]
if missing:
    raise RuntimeError(f"CDISC terms referenced but absent from the core T-Box: {missing}")
print(f"ok    CDISC terms referenced: {len(cdisc)}, none redeclared, none dangling")
for term in cdisc:
    print(f"        {term}")

still_dangling = sorted(map(str, undeclared_qbc_iris(written, ignore={VERSION_IRI})))
if still_dangling:
    raise RuntimeError(f"qbc: IRIs referenced but never declared: {still_dangling}")
print("ok    every qbc: IRI referenced in the graph is declared in it")

ok    classes declared: 9
ok    class-level skos:broadMatch -> https://www.cdisc.org/cosmos/biomedical_concept_v1.0/BiomedicalConcept  (present in core)
ok    CDISC terms referenced: 6, none redeclared, none dangling
        https://www.cdisc.org/cosmos/biomedical_concept_v1.0/BiomedicalConcept
        https://www.cdisc.org/cosmos/biomedical_concept_v1.0/BiomedicalConceptResultScaleEnum
        https://www.cdisc.org/cosmos/biomedical_concept_v1.0/DataElementConcept
        https://www.cdisc.org/cosmos/biomedical_concept_v1.0/definition
        https://www.cdisc.org/cosmos/biomedical_concept_v1.0/shortName
        https://www.cdisc.org/cosmos/sdtm_v1.0/AssignedTerm
ok    every qbc: IRI referenced in the graph is declared in it


## What this notebook does not produce

The JSON-LD context and the SHACL shapes for the overlay — the `40_` and `55_`
analogues — are not generated here. Both run cleanly against this schema
(measured 2026-09-01), and the shapes are what the conformance report will need,
but they are a separate step and are not part of this phase's deliverables.